In [1]:
import sys
import os
from transformers import BioGptTokenizer, BioGptForCausalLM
import torch

sys.path.append(os.path.abspath(".."))
from utils.preprocess import load_and_preprocess
from utils.llm_utils import LLMUtils

In [2]:
llmUtils = LLMUtils(("microsoft/BioGPT"))

In [3]:
# Initializing the model and tokenizer
tokenizer = BioGptTokenizer.from_pretrained("microsoft/BioGPT")
model = BioGptForCausalLM.from_pretrained("microsoft/BioGPT")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

In [4]:
# Setting the padding end-of-sequence as padding token if padding token is not there
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
torch.cuda.empty_cache()
has_gpu = torch.cuda.is_available()

device = torch.device("cuda" if has_gpu else "cpu") # My lap has GPU, but making it as dynamic, so you can run with cpu as well.
model.to(device)
model.eval()

BioGptForCausalLM(
  (biogpt): BioGptModel(
    (embed_tokens): BioGptScaledWordEmbedding(42384, 1024, padding_idx=1)
    (embed_positions): BioGptLearnedPositionalEmbedding(1026, 1024)
    (layers): ModuleList(
      (0-23): 24 x BioGptDecoderLayer(
        (self_attn): BioGptAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (activation_fn): GELUActivation()
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (final_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (layer_norm): LayerNorm((1024

In [6]:
# Load data
# train_df = load_and_preprocess('../Preprocessing+baseline/data/ori_pqal.json', False)
test_df = load_and_preprocess('../Preprocessing+baseline/data/test_set.json', False)

In [7]:
predicted_values = []
reference_values = []

In [ ]:
def generate_answer(question, context, max_new_tokens=2):
    prompt = llmUtils.build_prompt(question, context)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated = full_text[len(prompt):].strip()
    return generated

In [10]:
for index, row in test_df.iterrows():
    # normalizing the labels of the test data
    reference_values.append(llmUtils.normalize_label(row["label"]))

    # generating and normalizing the labels of the generated data
    generated_answer = generate_answer(row["question"], row["context"])
    predicted_values.append(llmUtils.normalize_label(generated_answer))

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


In [11]:
predicted_values

['no',
 'no',
 'no',
 'no',
 'no',
 'spinal subdural',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'r: if the',
 'no',
 ". answer:'",
 'yes',
 'yes',
 'wer: results suggest',
 'no',
 'no',
 'maybe',
 'no',
 'no',
 'mitochondrial dynamics',
 'no',
 'no',
 'r: the results',
 'yes',
 'r: question',
 'yes',
 'no',
 'yes',
 'r: the findings',
 'r: the findings',
 'no',
 'no',
 'yes',
 'yes',
 'the reasons',
 'yes',
 'no',
 'no',
 'this paper',
 'no',
 'r: a-',
 'heterotopic',
 '',
 'yes',
 'yes',
 ': this is',
 'no',
 'no',
 'r: fear of',
 'no',
 'no',
 'maybe',
 'yes',
 'yes',
 'no',
 'answer: amox',
 'maybe',
 'to clarify',
 'no',
 'yes',
 'no',
 'swer: this study',
 'no',
 'no',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 '"',
 'yes',
 'yes',
 'yes',
 'yes',
 'r: question',
 'no',
 'yes',
 'no',
 'n

In [12]:
accuracy = sum(p == r for p, r in zip(predicted_values, reference_values)) / len(reference_values)
print("Test accuracy:", accuracy)

Test accuracy: 0.34331337325349304
